# Distil the teacher on real receipts from Kaggle

The next SFT round, with real receipts from Kaggle added to the generated train suites: [dhiaznaidi/receiptdatasetssd300v2](https://www.kaggle.com/datasets/dhiaznaidi/receiptdatasetssd300v2) by default (`DATASET`). It follows `sft_a100.ipynb`, with these steps in front:

1. **Your Kaggle key, at run time.** Put it in Colab's secrets (the key icon on the left) as `KAGGLE_USERNAME` and `KAGGLE_KEY`, or as `KAGGLE_API_TOKEN` for a new-style `KGAT_…` token, and allow this notebook to read them. With no secret set, the cell asks for the key in a hidden prompt. Either way the key is passed only to the `kaggle` download command: it isn't written to disk, kept on Drive, or put in the environment of the other processes. The dataset's zip is kept on Drive, so a rerun doesn't ask again.
2. **Look at the data.** It prints what the dataset holds (file types, folders, the start of a few annotation files), because the converter has to find receipt text and labelled fields in it.
3. **Turn the receipts into suites.** `real_receipts.py` reads the usual receipt layouts: SROIE-style OCR box files with `key` labels, CORD-style JSON, CSV, Pascal VOC XML and COCO JSON. It pairs each receipt's text with its labelled merchant, total and date. Where the dataset has images and boxes but no text (an SSD300 detection set may be like that), it reads the text with tesseract (`OCR = "auto"`). Each receipt becomes one till case, with a simulated user around it:
   - The user is in the receipt's currency, has a ledger of past records drawn from other receipts in the same split, and has the categories those use.
   - For 70% of the cases the user has filed this merchant before, which decides the category. The rest need the category read from the receipt, so they are kept only where the merchant's name or text clearly says what kind of shop it is.
   - Receipts are dropped, and counted, when the labelled total isn't printed in the text, the date can't be read, several different times are printed, or the text duplicates another receipt.
   - The receipts are split by a hash of their text: 15% go to `dev-receipts`, which is evaluation only, and the rest to `train-receipts`, where each receipt goes to two different users.
   - Both suites are kept on Drive, so a rerun gets exactly the same cases even with OCR.
4. **The teacher on `train-receipts`, beside the generated train suites.** The teacher also runs on `dev-receipts`, for a reference row. Then come the pairs, SFT from the base model, the merge, and scores at bf16 on `dev`, `dev-novel` and `dev-receipts`. The shipped model (GRPO round 2) is scored on `dev-receipts` too when its merged copy is on Drive, so the before and after are on the same cases.

**Look at the converted cases before you start the teacher.** The labels come from the dataset, and the currency, the date order and the category come from the settings and from keyword rules. The *Convert* cell prints a few cases with their gold record. If one is wrong (for example a currency that is never printed, or US-style dates), change `CURRENCY`, `MONTH_FIRST` or `LANGUAGE`, delete `MyDrive/spending-agent/suites/train-receipts` and `dev-receipts`, and run the cell again. Changing them after the teacher has started is a new suite: `run.py` refuses to resume on a different fingerprint.

The receipts and the suites made from them stay on Drive and out of git (`suites/train*` and `suites/dev-receipts` are git-ignored), because the dataset's licence decides what may be published. The zip at the end holds only runs and counts, as in `sft_a100.ipynb`.

The GPU is the same as `sft_a100.ipynb`: **A100** while any teacher run is still to do, and an **L4** once the teacher's sessions are on Drive. A T4 doesn't work for SFT.

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
from google.colab import userdata

def sh(args, cwd=None, env=None):
    process = subprocess.Popen(args, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args[:3])


def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""


REPO_URL = "https://github.com/kurexi/cost"
BRANCH = "claude/modest-brahmagupta-hpbdfb"
RECORDS = "eval-records"
AGENT = ""
CLONE = "/content/repo"
EVAL = os.path.join(CLONE, AGENT, "eval")
RECORDS_DIR = f"{CLONE}/{RECORDS}" if RECORDS else EVAL
token = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{token}@", 1) if token else REPO_URL
os.chdir("/content")
sh(["rm", "-rf", CLONE])
sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CLONE])
os.environ["SPENDING_EVAL_RECORDS"] = RECORDS_DIR
os.makedirs(f"{RECORDS_DIR}/runs", exist_ok=True)

## Settings

**The dataset.** `DATASET` is the Kaggle dataset. Set `CURRENCY` for receipts that print no currency of their own: the converter recognises `RM`, `€`, `£`, `$`, `S$`, `Rp` and other markers, and uses `CURRENCY` only where it finds none. `MONTH_FIRST` reads `03/04/2019` as March 4th instead of April 3rd. `LANGUAGE` picks the category titles (`auto`: French if most receipts read French, otherwise English). With `OCR = "auto"`, tesseract runs only when the dataset gives no text; set `"always"` or `"never"` to force it. `OCR_LANGUAGES` lists tesseract's languages, for example `eng+fra`.

**The round.** `TEACHER_RUNS` lists the teacher runs the pairs are built from. Generated suites are regenerated and checked against their fingerprints; the real-receipt suite is marked `real` and comes from Drive. A run already finished on Drive is reused, so `teacher-train` and `teacher-train-2` cost nothing if round 4 made them. If `teacher-train-2` was never run, it costs about 2.6 hours on an A100; remove that entry to skip it. `ROUND` names the adapter, the merged model (`qwen3.5-0.8b-sft-r5`) and its runs. `PREVIOUS` and `SHIPPED` are what it is compared with. A run of them that isn't present is left out of the comparison.

In [ ]:
DATASET = "dhiaznaidi/receiptdatasetssd300v2"
CURRENCY = "USD"
MONTH_FIRST = False
LANGUAGE = "auto"
OCR = "auto"
OCR_LANGUAGES = "eng+fra"
SEED = 51
EVAL_SHARE = 0.15
TRAIN_COPIES = 2
TRAIN_SUITE = "train-receipts"
EVAL_SUITE = "dev-receipts"

TEACHER = "Qwen/Qwen3.8-27B-FP8"
TEACHER_RUNS = {
    "teacher-train": {"suite": "train", "fingerprint": "a5437ef135b8f48f", "cases": 2100,
                      "args": ["--per-locale", "300", "--users-per-locale", "30", "--seed", "1", "--novel-share", "0.5"]},
    "teacher-train-2": {"suite": "train-2", "fingerprint": "9c208ad58ed4d37a", "cases": 4200,
                        "args": ["--per-locale", "600", "--users-per-locale", "60", "--seed", "4", "--novel-share", "0.5"]},
    f"teacher-{TRAIN_SUITE}": {"suite": TRAIN_SUITE, "real": True},
}
STUDENT = "Qwen/Qwen3.5-0.8B"
ROUND = "r5"
PREVIOUS = "qwen3.5-0.8b-sft-r4"
EARLIER = "qwen3.5-0.8b-sft-r3"
SHIPPED = "qwen3.5-0.8b-grpo-r2"
LOSS = "token"
DOUBT_CONFIDENCE = 0.5
NAME = f"qwen3.5-0.8b-sft-{ROUND}"
TEACHER_NAME = "qwen3.8-27b-fp8-v5"
EPOCHS = 1
RANK = 16
LR = 2e-4
ACCUM = 8
MAX_LEN = 8192
print(NAME)

## Drive

In [ ]:
import json
import shutil
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/spending-agent"
DATA_ZIP = f"{DRIVE}/data/{DATASET.split('/')[-1]}.zip"
RAW = f"/content/data/{DATASET.split('/')[-1]}"
SUITES_KEPT = f"{DRIVE}/suites"
TEACHER_DIRS = {run: f"{DRIVE}/runs/{run}" for run in TEACHER_RUNS}
TEACHER_NOVEL_RUN = f"{TEACHER_NAME}-dev-novel"
TEACHER_REAL_RUN = f"{TEACHER_NAME}-{EVAL_SUITE}"
PAIRS = f"{DRIVE}/sft/{'+'.join(TEACHER_RUNS)}" + (f"-doubt{DOUBT_CONFIDENCE:g}" if DOUBT_CONFIDENCE is not None else "") + ".jsonl"
ADAPTER = f"{DRIVE}/sft/{NAME}"
MERGED = f"{DRIVE}/models/{NAME}"
SHIPPED_MODEL = f"{DRIVE}/models/{SHIPPED}"
for folder in (*TEACHER_DIRS.values(), f"{DRIVE}/runs/{TEACHER_NOVEL_RUN}", f"{DRIVE}/runs/{TEACHER_REAL_RUN}",
               os.path.dirname(DATA_ZIP), SUITES_KEPT, os.path.dirname(PAIRS), ADAPTER, os.path.dirname(MERGED)):
    os.makedirs(folder, exist_ok=True)


def keep_on_drive(run):
    kept, local = f"{DRIVE}/runs/{run}", f"{RECORDS_DIR}/runs/{run}"
    os.makedirs(kept, exist_ok=True)
    if os.path.islink(local):
        return
    if os.path.isdir(local) and not os.listdir(kept):
        shutil.copytree(local, kept, dirs_exist_ok=True)
    sh(["rm", "-rf", local])
    os.symlink(kept, local)


def finished(folder, cases):
    results = f"{folder}/results.jsonl"
    return os.path.exists(results) and sum(1 for _ in open(results)) >= cases

## The receipts from Kaggle

The zip is downloaded once into `MyDrive/spending-agent/data/` and unpacked to local disk on every run. Unpacking thousands of small files onto Drive would be slow.

The key comes from Colab's secrets (`KAGGLE_USERNAME` and `KAGGLE_KEY`, or `KAGGLE_API_TOKEN`). Without them, the cell asks for it in a hidden prompt. It is handed only to the `kaggle` command's environment.

In [ ]:
import getpass

if not os.path.exists(DATA_ZIP):
    username, key, api_token = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY"), secret("KAGGLE_API_TOKEN")
    if not api_token and not (username and key):
        print("No Kaggle secret in this notebook's Colab secrets: type it here instead.")
        typed = getpass.getpass("Kaggle API key, or a KGAT_ API token: ").strip()
        if typed.startswith("KGAT_"):
            api_token = typed
        else:
            username, key = input("Kaggle username: ").strip(), typed
    kaggle_env = {k: v for k, v in os.environ.items() if not k.startswith("KAGGLE_")}
    if api_token:
        kaggle_env["KAGGLE_API_TOKEN"] = api_token
    if username and key:
        kaggle_env.update(KAGGLE_USERNAME=username, KAGGLE_KEY=key)
    sh(["pip", "install", "-q", "-U", "kaggle"])
    download = "/content/kaggle-download"
    sh(["rm", "-rf", download])
    sh(["kaggle", "datasets", "download", DATASET, "-p", download], env=kaggle_env)
    del kaggle_env, username, key, api_token
    zips = [name for name in os.listdir(download) if name.endswith(".zip")]
    if len(zips) != 1:
        raise SystemExit(f"expected one zip from Kaggle, found {os.listdir(download)}")
    shutil.copy(f"{download}/{zips[0]}", DATA_ZIP)
    sh(["rm", "-rf", download])
if not os.path.isdir(RAW):
    os.makedirs(RAW)
    sh(["unzip", "-q", "-o", DATA_ZIP, "-d", RAW])
print(DATA_ZIP, f"{os.path.getsize(DATA_ZIP) / 1e6:.1f} MB")

## What the dataset holds

File types, the largest folders, and the start of one file of each annotation type. If the *Convert* cell keeps no receipts, this output says why. Paste it into a Claude Code session to have the converter taught the layout.

In [ ]:
from collections import Counter

extensions, folders, samples = Counter(), Counter(), {}
for folder, _, names in os.walk(RAW):
    for name in names:
        ext = os.path.splitext(name)[1].lower() or "(none)"
        extensions[ext] += 1
        folders[os.path.relpath(folder, RAW)] += 1
        if ext in (".txt", ".json", ".jsonl", ".csv", ".xml", ".tsv") and ext not in samples:
            samples[ext] = os.path.join(folder, name)
print("files by type:", dict(extensions.most_common()))
print("folders:")
for folder, count in folders.most_common(15):
    print(f"  {count:6d}  {folder}")
for ext, path in samples.items():
    print(f"\n== {os.path.relpath(path, RAW)}")
    with open(path, encoding="utf-8", errors="replace") as f:
        print(f.read(1200))

## Convert

The suites are built once and kept on Drive (`MyDrive/spending-agent/suites/`); a rerun copies them back. The counts say how many receipts were read, by which layout (`files:`), and why the others were dropped (`dropped:`). Three cases follow, with their gold records: check them before the teacher runs.

In [ ]:
os.chdir(EVAL)


def convert(ocr):
    args = ["python3", "real_receipts.py", "--source", RAW, "--dataset", DATASET, "--train-name", TRAIN_SUITE,
            "--eval-name", EVAL_SUITE, "--seed", str(SEED), "--eval-share", str(EVAL_SHARE), "--currency", CURRENCY,
            "--language", LANGUAGE, "--train-copies", str(TRAIN_COPIES), *(["--month-first"] if MONTH_FIRST else []),
            *(["--ocr", "--ocr-languages", OCR_LANGUAGES] if ocr else [])]
    done = subprocess.run(args, capture_output=True, text=True)
    print(done.stdout, done.stderr)
    return done.returncode == 0


def install_tesseract():
    languages = [l for l in OCR_LANGUAGES.split("+") if l != "eng"]
    sh(["apt-get", "-qq", "install", "-y", "tesseract-ocr", *[f"tesseract-ocr-{l}" for l in languages]])
    sh(["pip", "install", "-q", "pytesseract", "pillow"])


kept = [name for name in (TRAIN_SUITE, EVAL_SUITE) if os.path.exists(f"{SUITES_KEPT}/{name}/meta.json")]
if len(kept) == 2:
    for name in kept:
        shutil.copytree(f"{SUITES_KEPT}/{name}", f"suites/{name}", dirs_exist_ok=True)
    print("suites from Drive")
else:
    images = sum(n for ext, n in extensions.items() if ext in (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"))
    ok = OCR != "always" and convert(False)
    if ok and OCR == "auto" and images:
        with open(f"suites/{TRAIN_SUITE}/meta.json") as f:
            ok = json.load(f)["receipts"] >= 0.3 * images * (1 - EVAL_SHARE)
        if not ok:
            print("few receipts came with text: reading the images with tesseract instead")
    if not ok and OCR != "never" and images:
        install_tesseract()
        ok = convert(True)
    if not ok:
        raise SystemExit("no usable receipts: see the counts above and the output of the cell before")
    for name in (TRAIN_SUITE, EVAL_SUITE):
        shutil.copytree(f"suites/{name}", f"{SUITES_KEPT}/{name}", dirs_exist_ok=True)

SUITE_CASES = {}
for name in (TRAIN_SUITE, EVAL_SUITE):
    with open(f"suites/{name}/meta.json") as f:
        meta = json.load(f)
    SUITE_CASES[name] = meta["cases"]
    print(name, {k: meta[k] for k in ("cases", "users", "receipts", "fingerprint", "currencies", "language")})
print("conversion:", json.dumps(meta["conversion"], indent=1))
with open(f"suites/{EVAL_SUITE}/cases.jsonl", encoding="utf-8") as f:
    shown = [json.loads(line) for line in f][:3]
for case in shown:
    gold = case["gold"]
    print("\n" + "=" * 80 + f"\n{case['id']}  {case['traits']}\n" + case["content"][:1500])
    print("-> gold:", {k: gold[k] for k in ("merchant", "totalAmount", "currencyCode", "purchaseDate", "purchaseTime",
                                            "category")})

## The generated train suites

As in `sft_a100.ipynb`: each regenerated from its seed and checked against its fingerprint.

In [ ]:
os.chdir(EVAL)
for run, spec in TEACHER_RUNS.items():
    if spec.get("real"):
        spec["cases"] = SUITE_CASES[spec["suite"]]
        continue
    subprocess.run(["python3", "make_suite.py", "--name", spec["suite"], *spec["args"]], check=True, capture_output=True)
    with open(f"suites/{spec['suite']}/meta.json") as f:
        suite = json.load(f)
    print(spec["suite"], suite["cases"], "cases, fingerprint", suite["fingerprint"])
    if suite["fingerprint"] != spec["fingerprint"] or suite["cases"] != spec["cases"]:
        raise SystemExit(f"{spec['suite']} doesn't regenerate as it did: the generator changed, so this is a new suite")

for run in (*TEACHER_RUNS, TEACHER_NOVEL_RUN, TEACHER_REAL_RUN):
    keep_on_drive(run)
gpu_mib = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.split()[0])
TEACHER_FITS = gpu_mib >= 39000
NEEDS_TRAIN_RUNS = [] if os.path.exists(PAIRS) else [
    run for run, spec in TEACHER_RUNS.items() if not finished(TEACHER_DIRS[run], spec["cases"])]
NEEDS_EVAL_RUNS = [(run, suite) for run, suite, cases in ((TEACHER_NOVEL_RUN, "dev-novel", 280),
                                                          (TEACHER_REAL_RUN, EVAL_SUITE, SUITE_CASES[EVAL_SUITE]))
                   if not finished(f"{DRIVE}/runs/{run}", cases)]
if NEEDS_TRAIN_RUNS and not TEACHER_FITS:
    raise SystemExit(f"the teacher runs {NEEDS_TRAIN_RUNS} aren't finished and this GPU has {gpu_mib} MiB: use an A100")
SERVE_TEACHER = TEACHER_FITS and (bool(NEEDS_TRAIN_RUNS) or bool(NEEDS_EVAL_RUNS))
for run in TEACHER_RUNS:
    print(f"teacher run {run}:", "to run" if run in NEEDS_TRAIN_RUNS else "done")
for run, suite in NEEDS_EVAL_RUNS:
    print(f"teacher on {suite}:", "to run" if TEACHER_FITS else "missing, and this GPU can't host the teacher")

## vLLM

As in `sft_a100.ipynb`: its own virtual environment, the teacher's `dev` flags, logs in `/content/vllm.log`.

In [ ]:
VLLM_VENV = "/content/vllm-venv"
if not os.path.exists(f"{VLLM_VENV}/bin/vllm"):
    sh(["pip", "install", "-q", "uv"])
    sh(["uv", "venv", "-q", "--python", "3.12", VLLM_VENV])
    sh(["uv", "pip", "install", "-q", "--python", f"{VLLM_VENV}/bin/python", "vllm"])
print(subprocess.run([f"{VLLM_VENV}/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                     capture_output=True, text=True).stdout)

In [ ]:
import time
import urllib.request


def stop_servers():
    subprocess.run("pkill -f '[v]llm serve'; pkill -f '[l]lama-server'", shell=True)
    for _ in range(60):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.strip()
        if int(used.split()[0]) < 1000:
            return
        time.sleep(5)
    raise SystemExit(f"the GPU still holds {used} MiB")


def log_tail(path, characters=4000):
    with open(path, errors="replace") as f:
        return f.read()[-characters:]


def serve(model, extra=""):
    stop_servers()
    log_path = "/content/vllm.log"
    process = subprocess.Popen([f"{VLLM_VENV}/bin/vllm", "serve", model, "--served-model-name", "qwen",
                                "--max-model-len", "16384", "--max-num-seqs", "32", "--gpu-memory-utilization", "0.90",
                                "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
                                "--reasoning-parser", "qwen3", "--port", "8090", *extra.split()],
                               stdout=open(log_path, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    started = reported = time.time()
    while True:
        if process.poll() is not None:
            raise SystemExit(f"vLLM exited with code {process.returncode}:\n{log_tail(log_path)}")
        try:
            urllib.request.urlopen("http://127.0.0.1:8090/v1/models", timeout=5)
            print(f"server is up after {time.time() - started:.0f} s")
            return
        except Exception:
            pass
        if time.time() - started > 2400:
            raise SystemExit(f"vLLM isn't up after 40 minutes:\n{log_tail(log_path)}")
        if time.time() - reported >= 30:
            lines = log_tail(log_path, 1000).strip().splitlines()
            print(f"{time.time() - started:.0f} s, still starting: {lines[-1] if lines else '(no log yet)'}", flush=True)
            reported = time.time()
        time.sleep(2)


RUN_ARGS = ["--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen", "--reasoning-effort", "none", "--workers", "16"]

## The teacher, with the harness checked first

On `train-receipts`: `oracle` must grade 1.000 and `silent` 0, and the server must accept the tool round and the full bounded schema. Then come the train runs still to do, the teacher on `dev-novel` if its row is missing, and the teacher on `dev-receipts`. Every run resumes from Drive after a disconnect. `train-receipts` has about 1,700 cases, about 40 minutes on an A100.

In [ ]:
os.chdir(EVAL)
if SERVE_TEACHER:
    serve(TEACHER)
    for name, backend in (("check-oracle", "oracle"), ("check-silent", "silent")):
        graded = subprocess.run(["python3", "run.py", "--suite", TRAIN_SUITE, "--name", name, "--backend", backend,
                                 "--limit", "70", "--restart"], capture_output=True, text=True).stdout
        print(name, [line for line in graded.splitlines() if line.startswith("record_core")])
    probe = subprocess.run(["python3", "probe_formats.py", "--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen"],
                           capture_output=True, text=True)
    print(probe.stdout, probe.stderr)
    verdicts = dict((line[:32].strip(), line[32:].strip()) for line in probe.stdout.splitlines() if line.strip())
    for required in ("tool round (tool_choice auto)", "full bounded schema"):
        if not verdicts.get(required, "").startswith("accepted"):
            raise SystemExit(f"the server refused the {required}: stop here")
    for run in NEEDS_TRAIN_RUNS:
        sh(["python3", "run.py", "--suite", TEACHER_RUNS[run]["suite"], "--name", run, *RUN_ARGS])
    for run, suite in NEEDS_EVAL_RUNS:
        sh(["python3", "run.py", "--suite", suite, "--name", run, *RUN_ARGS])
else:
    print("the teacher isn't needed" if not NEEDS_EVAL_RUNS else "the teacher's evaluation rows are missing: use an A100")

## The shipped model on `dev-receipts`

The before for this round: GRPO round 2 at bf16, from its merged copy on Drive (`MyDrive/spending-agent/models/qwen3.5-0.8b-grpo-r2`, made by `grpo.ipynb`). Skipped if that copy isn't there. About 3 minutes.

In [ ]:
os.chdir(EVAL)
SHIPPED_RUN = f"{SHIPPED}-bf16-{EVAL_SUITE}"
keep_on_drive(SHIPPED_RUN)
if finished(f"{RECORDS_DIR}/runs/{SHIPPED_RUN}", SUITE_CASES[EVAL_SUITE]):
    print("done already")
elif os.path.exists(f"{SHIPPED_MODEL}/config.json"):
    serve(SHIPPED_MODEL, "--dtype bfloat16")
    sh(["python3", "run.py", "--suite", EVAL_SUITE, "--name", SHIPPED_RUN, *RUN_ARGS])
else:
    print(f"no merged model at {SHIPPED_MODEL}: skipped")
stop_servers()
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## The training pairs

`sft_data.py` over every run in `TEACHER_RUNS`, as in `sft_a100.ipynb`. Its counts, written beside the pairs, show how many of the real-receipt sessions the teacher got right (`kept`) and how many come back as doubt pairs.

In [ ]:
!pip install -q -U transformers peft flash-linear-attention
!pip uninstall -q -y torchao
from huggingface_hub import snapshot_download

BASE = "/content/models/qwen35-08b"
snapshot_download(STUDENT, local_dir=BASE)
%cd {EVAL}
if not os.path.exists(PAIRS):
    sh(["python3", "sft_data.py", *TEACHER_RUNS, "--template", BASE, "--tokenizer", BASE,
        "--verify-template", "--max-len", str(MAX_LEN), "--out", PAIRS,
        *(["--doubt-confidence", str(DOUBT_CONFIDENCE)] if DOUBT_CONFIDENCE is not None else [])])
print(open(PAIRS.replace(".jsonl", ".stats.json")).read())

## SFT

A fresh LoRA from the base model on all the pairs, as in `sft_a100.ipynb`. It checkpoints to Drive every 50 steps and resumes when rerun.

In [ ]:
%cd {EVAL}
if not os.path.exists(f"{ADAPTER}/adapter_model.safetensors"):
    sh(["python3", "train_sft.py", "--model", BASE, "--data", PAIRS,
        "--out", ADAPTER, "--epochs", str(EPOCHS), "--rank", str(RANK), "--lr", str(LR),
        "--accum", str(ACCUM), "--max-len", str(MAX_LEN), "--loss", LOSS, "--save-every", "50"])
!tail -n 3 {ADAPTER}/train_log.jsonl

## Merge

In [ ]:
%cd {EVAL}
if not os.path.exists(f"{MERGED}/merge.json"):
    sh(["python3", "merge_lora.py", "--base", BASE, "--adapter", ADAPTER, "--out", MERGED])
!ls -la {MERGED}

## The merged model on `dev`, `dev-novel` and `dev-receipts`

At bf16 on vLLM, each suite compared with the earlier rounds, the shipped model and the teacher (the first one present is the baseline). `dev-receipts` says whether real receipts got better, and `dev` and `dev-novel` say whether anything else got worse.

In [ ]:
serve(MERGED, "--dtype bfloat16")
os.chdir(EVAL)
SCORED = {"dev": f"{NAME}-bf16", "dev-novel": f"{NAME}-bf16-dev-novel", EVAL_SUITE: f"{NAME}-bf16-{EVAL_SUITE}"}
for suite, run in SCORED.items():
    keep_on_drive(run)
    sh(["python3", "run.py", "--suite", suite, "--name", run, *RUN_ARGS])
stop_servers()
LINEUPS = {
    "dev": [f"{EARLIER}-bf16", f"{PREVIOUS}-bf16", f"{SHIPPED}-bf16", SCORED["dev"], TEACHER_NAME],
    "dev-novel": [f"{EARLIER}-bf16-dev-novel", f"{PREVIOUS}-bf16-dev-novel", f"{SHIPPED}-bf16-dev-novel",
                  SCORED["dev-novel"], TEACHER_NOVEL_RUN],
    EVAL_SUITE: [SHIPPED_RUN, SCORED[EVAL_SUITE], TEACHER_REAL_RUN],
}
for suite, lineup in LINEUPS.items():
    cases = SUITE_CASES.get(suite, 280)
    present = [run for run in lineup if finished(f"{RECORDS_DIR}/runs/{run}", cases)]
    print(f"\n#### {suite}: {present}")
    if len(present) >= 2:
        sh(["python3", "compare.py", *present])
    for run in present:
        with open(f"{RECORDS_DIR}/runs/{run}/grade.json") as f:
            per_case = json.load(f)["per_case"]
        per_case = list(per_case.values()) if isinstance(per_case, dict) else per_case
        approved = [c for c in per_case if c["would_approve"]]
        print(f"{run}: approval precision {sum(c['record_all'] for c in approved) / max(1, len(approved)):.2f} "
              f"over {len(approved)} approved")
sh(["python3", "board.py", "--write"])

## Take the results home

As in `sft_a100.ipynb`: for each run, its `meta.json`, `results.jsonl` and `grade.json`, plus the pair counts, the adapter's config and train log, the merge report and `RESULTS.md`. The receipts, the suites and the transcripts stay on Drive. Before committing the `dev-receipts` and `train-receipts` runs, check what the dataset's licence allows: `results.jsonl` holds the drafts, which quote the receipts' merchants.

In [ ]:
OUT = "/content/out"
shutil.rmtree(OUT, ignore_errors=True)
for run, folder in TEACHER_DIRS.items():
    os.makedirs(f"{OUT}/runs/{run}")
    for name in ("meta.json", "results.jsonl", "grade.json"):
        shutil.copy(f"{folder}/{name}", f"{OUT}/runs/{run}/")
os.makedirs(f"{OUT}/sft/{NAME}")
for source in (PAIRS.replace(".jsonl", ".stats.json"), f"{ADAPTER}/adapter_config.json", f"{ADAPTER}/train_args.json",
               f"{ADAPTER}/train_log.jsonl", f"{MERGED}/merge.json"):
    shutil.copy(source, f"{OUT}/sft/{NAME}/")
for run in (TEACHER_NOVEL_RUN, TEACHER_REAL_RUN, SHIPPED_RUN, *SCORED.values()):
    folder = f"{RECORDS_DIR}/runs/{run}"
    if finished(folder, 1):
        os.makedirs(f"{OUT}/runs/{run}")
        for name in ("meta.json", "results.jsonl", "grade.json"):
            if os.path.exists(f"{folder}/{name}"):
                shutil.copy(f"{folder}/{name}", f"{OUT}/runs/{run}/")
for name in (TRAIN_SUITE, EVAL_SUITE):
    os.makedirs(f"{OUT}/suites/{name}")
    shutil.copy(f"suites/{name}/meta.json", f"{OUT}/suites/{name}/")
shutil.copy(f"{RECORDS_DIR}/RESULTS.md", OUT)
subprocess.run(f"cd {OUT} && zip -qr /content/{NAME}.zip .", shell=True, check=True)
from google.colab import files
files.download(f"/content/{NAME}.zip")